# Day 9 — Exploratory Data Analysis (EDA)
### LimeX AI/ML Foundation Internship

**Goal:** Understand distributions, patterns, relationships, source behaviour, content quality, and time-based trends in the cleaned news dataset before feature engineering or machine learning.

**Input:** `news_clean.csv` (Day 7 preprocessing output)
**New columns added (per Day 9 instructions):**
- `category` — one of: **health, politics, stock, market, energy, technology, other** (assigned via keyword matching)
- `domain` — extracted from `news_url_clean`

**EDA principle:** *Describe and investigate — do not silently modify or delete records.*

## Learning Outcomes
After completing this notebook you will be able to:
1. Profile a real-world dataset: schema, missingness, duplicates, memory.
2. Visualise text-length distributions and interpret extraction quality.
3. Analyse source and category imbalance.
4. Extract temporal patterns from publication timestamps.
5. Detect suspicious/failed extractions using heuristics.
6. Write structured, evidence-based EDA findings.

## 1. Setup — Libraries & Configuration

In [1]:
from __future__ import annotations
import json, sys, re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")           # change to "inline" in a live Jupyter session
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

HERE       = Path().resolve()
REPO_ROOT  = HERE.parent.parent   # Limox/

OUTPUT_DIR = HERE / "eda_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Category keywords (from Day 9 instructions, 1.txt)
CATEGORY_KEYWORDS = {
    "health":     ["health","medical","hospital","disease","drug","vaccine","pandemic","doctor","patient","pharma"],
    "politics":   ["politics","election","government","president","parliament","senate","vote","policy","minister"],
    "stock":      ["stock","shares","equity","nasdaq","nyse","ipo","dividend","earnings"],
    "market":     ["market","economy","gdp","inflation","trade","export","import","fiscal","monetary"],
    "energy":     ["energy","oil","gas","coal","solar","wind","renewable","power","electricity","fuel"],
    "technology": ["technology","tech","ai","machine learning","software","hardware","startup","cyber","cloud","digital"],
}
print("Libraries loaded. Output directory:", OUTPUT_DIR)

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load Dataset

In [ ]:
CANDIDATE_PATHS = [
    REPO_ROOT / "Pre2-Talha-Saleem" / "Task 5 data validation and quality assessment" / "validation_output" / "news_accepted.csv",
    REPO_ROOT / "Pre2-Talha-Saleem" / "Task 4 data preprocessing pipeline" / "preprocessed_output" / "news_clean.csv",
    REPO_ROOT / "day 4" / "output_bulk" / "final" / "articles_bulk_new.csv",
]

source_path = None
for p in CANDIDATE_PATHS:
    if p.exists():
        source_path = p
        break

if source_path is None:
    sys.exit("No input CSV found. Run Day 7 preprocessing first.")

df = pd.read_csv(source_path)
print(f"Loaded: {source_path.name}")
print(f"Shape:  {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 3. Detect & Map Column Names

Our pipeline uses cleaned column names (e.g. `title_clean`, `content_clean`).
This cell maps them to canonical keys so all later code is column-name-agnostic.

In [ ]:
def pick(cols_set, *candidates):
    for c in candidates:
        if c in cols_set: return c
    return None

cols_set = set(df.columns)
COLS = {
    "url":       pick(cols_set, "news_url_clean", "news_url", "url"),
    "title":     pick(cols_set, "title_clean", "title"),
    "content":   pick(cols_set, "content_clean", "content", "words"),
    "author":    pick(cols_set, "author_clean", "author"),
    "published": pick(cols_set, "published_at_utc", "published_time", "date"),
    "scraped":   pick(cols_set, "scraped_at"),
    "domain":    pick(cols_set, "domain", "source"),
    "category":  pick(cols_set, "category", "article_niche", "channel"),
}
print("Detected field mapping:")
for k, v in COLS.items():
    print(f"  {k:12s} -> {v}")

## 4. Add `category` and `domain` Columns (Day 9 Requirement)

**Day 9 instructions (1.txt)** specify adding:
- A `category` column — one of: health, politics, stock, market, energy, technology
- A `domain` column — extracted from the article URL

We assign category by matching keywords in the title and content text.

In [ ]:
# 4a. Domain from URL
if not COLS["domain"] and COLS["url"]:
    def extract_domain(u):
        if not isinstance(u, str): return None
        m = re.search(r"(?:https?://)?(?:www\.)?([^/\s?#]+)", u)
        return m.group(1).lower() if m else None
    df["domain"] = df[COLS["url"]].apply(extract_domain)
    COLS["domain"] = "domain"
    print("domain column created from URL")

# 4b. Category from keywords
if not COLS["category"]:
    def assign_category(row):
        text = " ".join(filter(None, [
            str(row.get(COLS["title"] or "", "") or ""),
            str(row.get(COLS["content"] or "", "") or ""),
        ])).lower()
        for cat, kws in CATEGORY_KEYWORDS.items():
            if any(kw in text for kw in kws): return cat
        return "other"
    print("Assigning categories (this may take ~30s on 70k rows)...")
    df["category"] = df.apply(assign_category, axis=1)
    COLS["category"] = "category"
    print("category column created")

print("\nCategory value counts:")
print(df[COLS["category"]].value_counts())

## 5. Add Text-Length Feature Columns

In [ ]:
# Use precomputed columns from Day 7 where available
if "title_word_count" in df.columns:
    df["_title_words"] = df["title_word_count"]
    df["_title_chars"] = df.get("title_char_count", pd.Series(0, index=df.index))
elif COLS["title"]:
    df["_title_words"] = df[COLS["title"]].astype("string").str.split().str.len().fillna(0).astype(int)
    df["_title_chars"] = df[COLS["title"]].astype("string").str.len().fillna(0).astype(int)

if "content_word_count" in df.columns:
    df["_content_words"] = df["content_word_count"]
    df["_content_chars"] = df.get("content_char_count", pd.Series(0, index=df.index))
elif COLS["content"]:
    df["_content_words"] = df[COLS["content"]].astype("string").str.split().str.len().fillna(0).astype(int)
    df["_content_chars"] = df[COLS["content"]].astype("string").str.len().fillna(0).astype(int)

print("Text length columns ready.")
print(df[["_title_words","_content_words"]].describe(percentiles=[.05,.5,.95]).round(1))

## 6. Dataset Overview & Schema

Inspect shape, data types, non-null counts, unique values, and sample values for every column.

In [ ]:
overview = pd.DataFrame({
    "dtype":       df.dtypes.astype(str),
    "non_null":    df.notna().sum(),
    "missing":     df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique":      df.nunique(),
    "sample":      [str(df[c].dropna().iloc[0])[:60] if df[c].notna().any() else "" for c in df.columns],
})

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]:,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
overview.sort_values("missing_pct", ascending=False)

## 7. Missingness Analysis

Missing values may carry information — author or publication time absent more often for certain sources.

In [ ]:
missing_summary = overview[["missing","missing_pct"]].sort_values("missing_pct", ascending=False)
to_plot = missing_summary[missing_summary["missing_pct"] > 0]

if len(to_plot):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(to_plot.index, to_plot["missing_pct"], color="#C44E52")
    ax.set_title("Missing values by column (%)")
    ax.set_ylabel("Missing (%)")
    ax.set_ylim(0, 100)
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "missing_values.png", dpi=100)
    plt.show()
else:
    print("No missing values detected — dataset is complete.")

In [ ]:
# Per-domain missingness breakdown
if COLS["domain"] and COLS["author"] and COLS["published"]:
    tmp = df.copy()
    tmp["_dg"] = tmp[COLS["domain"]].fillna("(none)")
    records = []
    for dom, grp in tmp.groupby("_dg", dropna=False):
        records.append({
            "domain":                dom,
            "articles":              len(grp),
            "author_missing_pct":    round(grp[COLS["author"]].isna().mean()*100, 1),
            "published_missing_pct": round(grp[COLS["published"]].isna().mean()*100, 1),
        })
    dom_miss = pd.DataFrame(records).sort_values("articles", ascending=False)
    display(dom_miss.head(20))

## 8. Duplicate Analysis

Exact row duplicates, duplicate URLs, repeated titles, and repeated bodies are **different signals**.
Treat them as candidates to investigate, not automatic deletion instructions.

In [ ]:
duplicate_stats = {
    "exact_duplicate_rows":   int(df.duplicated().sum()),
    "duplicate_url_rows":     int(df[COLS["url"]].duplicated(keep=False).sum()) if COLS["url"] else None,
    "duplicate_title_rows":   int(df[COLS["title"]].astype("string").str.strip().duplicated(keep=False).sum()) if COLS["title"] else None,
    "duplicate_content_rows": int(df[COLS["content"]].astype("string").str.strip().duplicated(keep=False).sum()) if COLS["content"] else None,
}
display(pd.Series(duplicate_stats, name="row_count").to_frame())

In [ ]:
# Sample of duplicate-title articles
if COLS["title"]:
    title_dupes = df[df[COLS["title"]].astype("string").str.strip().duplicated(keep=False)]
    view_cols = [c for c in [COLS["domain"], COLS["published"], COLS["title"], COLS["url"]] if c]
    print(f"Rows with a duplicated title: {len(title_dupes):,}")
    display(title_dupes[view_cols].sort_values(COLS["title"]).head(10))

## 9. Text-Length Distributions

Article length is both a content characteristic and an extraction diagnostic.
Very short text may indicate failed extraction; very long text may contain navigation / comments.

In [ ]:
length_cols = [c for c in ["_title_chars","_title_words","_content_chars","_content_words"] if c in df.columns]
display(df[length_cols].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T.round(1))

In [ ]:
for col in length_cols:
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    if not len(vals): continue
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(vals.clip(upper=vals.quantile(.99)), bins=40, color="#4C72B0", edgecolor="white")
    ax.set_title(f"Distribution: {col}  (clipped at 99th percentile)")
    ax.set_xlabel(col); ax.set_ylabel("Articles")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"dist_{col}.png", dpi=100)
    plt.show()

In [ ]:
# Shortest and longest articles
if "_content_words" in df.columns:
    view = [c for c in [COLS["domain"], COLS["title"], COLS["published"], "_content_words", COLS["url"]] if c]
    print("=== 10 shortest articles ===")
    display(df.nsmallest(10, "_content_words")[view])
    print("\n=== 10 longest articles ===")
    display(df.nlargest(10, "_content_words")[view])

## 10. Domain / Source Distribution

Inspect which domains contribute most, whether the data is concentrated in a few sites,
and how article lengths vary by source.

In [ ]:
if COLS["domain"]:
    source_counts = df[COLS["domain"]].fillna("(none)").value_counts()
    print("Top 10 domains:")
    display(source_counts.rename("articles").to_frame().head(10))
    print(f"\nTop-1 domain share: {source_counts.iloc[0]/source_counts.sum()*100:.1f}%")
    print(f"Top-5 domain share: {source_counts.head(5).sum()/source_counts.sum()*100:.1f}%")

In [ ]:
if COLS["domain"]:
    fig, ax = plt.subplots(figsize=(11, 6))
    source_counts.head(20).sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
    ax.set_title("Top 20 domains by article count")
    ax.set_xlabel("Articles")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "domain_distribution.png", dpi=100)
    plt.show()

In [ ]:
# Content length by domain
if COLS["domain"] and "_content_words" in df.columns:
    src_len = (df.groupby(COLS["domain"], dropna=False)
               .agg(articles=("_content_words","size"),
                    median_words=("_content_words","median"),
                    mean_words=("_content_words","mean"),
                    p95_words=("_content_words", lambda s: s.quantile(.95)))
               .query("articles >= 3")
               .sort_values("articles", ascending=False)
               .round(1))
    display(src_len.head(20))

## 11. Category Distribution

Categories were assigned by keyword matching on title + content.
Class imbalance matters for classification and recommendation models.

In [ ]:
if COLS["category"]:
    cat_counts = df[COLS["category"]].fillna("other").value_counts()
    display(cat_counts.rename("articles").to_frame())
    fig, ax = plt.subplots(figsize=(10, 5))
    cat_counts.sort_values().plot(kind="barh", ax=ax, color="#55A868")
    ax.set_title("Article count by category")
    ax.set_xlabel("Articles")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "category_distribution.png", dpi=100)
    plt.show()

In [ ]:
# Category mix by domain (stacked bar)
if COLS["domain"] and COLS["category"] and df[COLS["category"]].nunique() <= 20:
    mix = pd.crosstab(df[COLS["domain"]], df[COLS["category"]], normalize="index") * 100
    display(mix.round(1).head(15))
    fig2, ax2 = plt.subplots(figsize=(12, max(4, len(mix.head(15)) * 0.45)))
    mix.head(15).plot(kind="bar", stacked=True, ax=ax2)
    ax2.set_title("Category composition by domain (%)")
    ax2.set_ylabel("Share within domain (%)")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "category_by_domain.png", dpi=100)
    plt.show()

## 12. Publication-Time Patterns

Convert timestamps to UTC explicitly. Invalid/missing timestamps become `NaT` and are counted.
Explore coverage, articles per day/hour, weekday patterns, and publication-to-scrape delay.

In [ ]:
if COLS["published"]:
    df["_pub_dt"] = pd.to_datetime(df[COLS["published"]], errors="coerce", utc=True)
    valid = df["_pub_dt"].dropna()
    print(f"Valid publication timestamps: {len(valid):,} / {len(df):,}")
    if len(valid):
        print(f"Earliest: {valid.min()}  |  Latest: {valid.max()}")
        print(f"Coverage: {round((valid.max()-valid.min()).total_seconds()/86400, 1)} days")
        df["_pub_date"] = df["_pub_dt"].dt.floor("D")
        df["_pub_hour"] = df["_pub_dt"].dt.hour
        df["_pub_day"]  = df["_pub_dt"].dt.day_name()

In [ ]:
# Daily article count
if "_pub_date" in df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    df.groupby("_pub_date").size().plot(ax=ax, color="#C44E52")
    ax.set_title("Articles per publication day (UTC)")
    ax.set_xlabel("Date"); ax.set_ylabel("Articles")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "articles_per_day.png", dpi=100)
    plt.show()

In [ ]:
# Hourly & weekday patterns
if "_pub_hour" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    # hour
    hr = df["_pub_hour"].value_counts().sort_index().reindex(range(24), fill_value=0)
    hr.plot(kind="bar", ax=axes[0], color="#8172B2")
    axes[0].set_title("Articles by hour (UTC)"); axes[0].set_xlabel("Hour")
    # weekday
    order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    wd = df["_pub_day"].value_counts().reindex(order, fill_value=0)
    wd.plot(kind="bar", ax=axes[1], color="#DD8452")
    axes[1].set_title("Articles by weekday (UTC)"); axes[1].set_xlabel("Day")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "articles_per_hour_weekday.png", dpi=100)
    plt.show()

In [ ]:
# Publish-to-scrape delay
if COLS["published"] and COLS["scraped"]:
    df["_scr_dt"] = pd.to_datetime(df[COLS["scraped"]], errors="coerce", utc=True)
    df["_delay_m"] = (df["_scr_dt"] - df["_pub_dt"]).dt.total_seconds() / 60
    print("Delay statistics (minutes):")
    display(df["_delay_m"].describe(percentiles=[.01,.05,.5,.95,.99]).to_frame().round(1))
    print("Negative delays (published AFTER scrape):", int((df["_delay_m"]<0).sum()))
    delay = df["_delay_m"].dropna()
    cl = delay.clip(lower=delay.quantile(.01), upper=delay.quantile(.99))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(cl, bins=40, color="#4C72B0")
    ax.set_title("Publish-to-scrape delay — clipped 1st-99th pct")
    ax.set_xlabel("Minutes"); ax.set_ylabel("Articles")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "scrape_delay.png", dpi=100)
    plt.show()
else:
    print("Both published and scraped_at required for delay analysis.")

## 13. Relationships Between Variables

Compare title length with content length and inspect category mix by domain.
Correlation is descriptive, not causal.

In [ ]:
if "_title_words" in df.columns and "_content_words" in df.columns:
    pair = df[["_title_words","_content_words"]].dropna()
    if len(pair) >= 2:
        r_pearson  = round(pair.corr(method="pearson").iloc[0,1], 3)
        r_spearman = round(pair.corr(method="spearman").iloc[0,1], 3)
        print(f"Pearson correlation:  {r_pearson}")
        print(f"Spearman correlation: {r_spearman}")
        samp = pair.sample(min(len(pair), 5000), random_state=42)
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(samp["_title_words"], samp["_content_words"], alpha=.4, s=8, color="#4C72B0")
        ax.set_title(f"Title words vs content words  (Pearson r = {r_pearson})")
        ax.set_xlabel("Title words"); ax.set_ylabel("Content words")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "title_vs_content.png", dpi=100)
        plt.show()

## 14. Suspicious Extraction Patterns

Heuristics flag candidates for **manual review**. Thresholds are starting points.
Do NOT automatically delete short or long articles.

In [ ]:
suspicious = pd.Series(False, index=df.index)

if "_content_words" in df.columns:
    suspicious |= df["_content_words"].fillna(0).lt(40)
    suspicious |= df["_content_words"].gt(df["_content_words"].quantile(.99))

if COLS["title"]:
    bad = r"access denied|captcha|robot check|page not found|403|enable javascript"
    suspicious |= df[COLS["title"]].astype("string").str.contains(bad, case=False, na=False)

if "_delay_m" in df.columns:
    suspicious |= df["_delay_m"].lt(0).fillna(False)

view = [c for c in [COLS["domain"], COLS["title"], COLS["published"], "_title_words", "_content_words", COLS["url"]] if c and c in df.columns]
suspicious_records = df.loc[suspicious, view].copy()

pct = len(suspicious_records) / max(len(df), 1) * 100
print(f"Suspicious candidates: {len(suspicious_records):,} / {len(df):,} ({pct:.1f}%)")
display(suspicious_records.head(30))

## 15. Numeric Correlation Matrix

In [ ]:
numeric = df.select_dtypes(include=np.number)
keep   = [c for c in numeric.columns if not c.startswith("Unnamed")]
numeric = numeric[keep]

if numeric.shape[1] >= 2:
    corr = numeric.corr(numeric_only=True)
    n = len(corr.columns)
    fig, ax = plt.subplots(figsize=(max(6, n*.7), max(5, n*.6)))
    im = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(n)); ax.set_xticklabels(corr.columns, rotation=60, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels(corr.index)
    ax.set_title("Numeric feature correlation matrix")
    fig.colorbar(im, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=100)
    plt.show()
else:
    print("Fewer than 2 numeric columns.")

## 16. Export EDA Outputs

Summaries and candidate records are exported. The original dataset is **not overwritten**.

In [ ]:
overview.to_csv(OUTPUT_DIR / "column_profile.csv")
missing_summary.to_csv(OUTPUT_DIR / "missingness_summary.csv")
pd.Series(duplicate_stats, name="row_count").to_csv(OUTPUT_DIR / "duplicate_summary.csv")

if "source_counts" in dir():
    source_counts.rename("articles").to_csv(OUTPUT_DIR / "source_distribution.csv")
if "cat_counts" in dir():
    cat_counts.rename("articles").to_csv(OUTPUT_DIR / "category_distribution.csv")

suspicious_records.to_csv(OUTPUT_DIR / "suspicious_candidates.csv", index=False)

summary = {
    "rows":                  int(len(df)),
    "columns":               int(df.shape[1]),
    "exact_duplicate_rows":  duplicate_stats.get("exact_duplicate_rows"),
    "suspicious_candidates": int(len(suspicious_records)),
    "missing_pct_by_column": {str(k):float(v) for k,v in overview["missing_pct"].items()},
    "detected_fields":       {k:v for k,v in COLS.items()},
    "outputs_directory":     str(OUTPUT_DIR.resolve()),
}
with open(OUTPUT_DIR / "eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved to:", OUTPUT_DIR.resolve())
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 17. How to Write EDA Findings

For each finding, record:

| Field | What to write |
|---|---|
| **Observation** | What does the chart/table show? |
| **Evidence** | Count, percentage, median, range, or comparison |
| **Interpretation** | What might explain the pattern? |
| **Caveat** | What else could explain it? |
| **Action** | Investigate, validate, collect more data, or design a feature |

> **Example (illustrative only):** "One source contributes 38% of rows. This may overrepresent that publisher, although it could reflect genuine publishing volume or the scraper schedule. Compare source coverage over time before deciding whether source balancing is needed."

## 18. Student Exercises

1. Find the top 3 domains and calculate their combined share of articles.
2. Compare median content length across the 5 largest domains.
3. Which category is most frequent in your dataset? What keyword drove it?
4. Plot daily article counts and identify any gaps or spikes; suggest two explanations.
5. Are duplicate titles from the same domain or different domains? What does that suggest?
6. Select one suspicious record and document why it deserves review.
7. Write 5 structured findings using the observation → evidence → interpretation → caveat → action format.
8. Why is automatically deleting every short article unsafe?

## Key Takeaways

- EDA reveals structure, distributions, missingness, duplicates, source concentration, temporal patterns, and relationships.
- Source and time analyses can expose collection bias and scraper reliability issues.
- Suspicious rows are **review candidates**, not automatically invalid records.
- Keep EDA separate from data cleaning and log evidence-based findings.

**Next:** Feature Engineering for News/Text Data — converting validated articles into numerical and categorical signals for ML.